# Graph-Laplacian regression on the Colorado basin — fixed version + Kron vs full check

This notebook is a corrected companion to `largest_uncompressed_laplacian.ipynb`, which is left unchanged.
It covers two things.

**1. Kron vs full Laplacian.** For each of the five MERIT edge-weighting schemes, on both the uncompressed
(67,401-node) and the compressed (67-node) graph, it:

* builds the full Laplacian `L = D − W` and its Kron (Schur-complement) reduction onto the 26 WQ nodes;
* checks they agree on everything Kron reduction must preserve exactly: effective resistance between WQ nodes,
  the Dirichlet energy of harmonic extensions, and Kron(uncompressed) vs Kron(compressed);
* re-runs the nested leave-one-node-out graph regression with **four** Laplacians and compares R²:

| variant | Laplacian in the model | size |
|---|---|---|
| `kron_uncompressed` | Kron of the 67k graph (what the original notebook uses) | 26 × 26 |
| `kron_compressed` | Kron of the 67-node graph | 26 × 26 |
| `full_uncompressed` | the full 67k Laplacian, graph effect g on **every** node | 67,401 × 67,401 |
| `full_compressed` | the full 67-node Laplacian, g on every node | 67 × 67 |

The `full_*` fits never form a Schur complement: they solve the complete (β, g) system with a sparse
factorisation. Each full Laplacian is divided by the same scale as its Kron reduction, so λ means the same thing.

**2. Fixes to problems in the original notebook.**

| # | Problem in the original | Fix here |
|---|---|---|
| 1 | pH-transform section (cells 63–67) used the **unscaled** Kron Laplacian, so its λ was ~21× weaker than everywhere else | uses the scaled Laplacian; the unscaled one is run alongside to show the difference (§7) |
| 2 | `REDUCED_STATIC`, `DO_BASE_FEATURES` and `COND_BASE_FEATURES` were picked from outer-LOO results, then re-scored with the same outer LOO (optimistic R²) | feature set and Fourier K chosen **inside** the inner LOO; the biased "best outer" number is shown beside the honest one (§8) |
| 3 | the "discharge" edge feature is an exact copy of upstream area (`log Q = a + b·log UPA`) | discharge dropped (§3) |
| 4 | HAND is 0 at every river pixel, so it adds nothing as a predictor | checked and dropped (§2) |
| 5 | comparison tables used hard-coded R² values from earlier runs | every table is computed from this run's results |
| 6 | the discharge fit used median discharge from held-out nodes | removed together with discharge (§3) |

### Run time
| section | approx. time |
|---|---|
| §1–§4 data, Laplacians and matrix checks | < 1 min |
| §6 Kron vs full R², without the 67k-node fits | ~3 min |
| §6 Kron vs full R², **with** the 67k-node fits (`RUN_FULL_67K = True`) | ~40 min |
| §7 pH transforms | ~2 min |
| §8 nested feature selection | ~15 min |

To skip the slow 67k fits, set `RUN_FULL_67K = False` below. Environment variables for a quick smoke test:
`LAPNB_TARGETS=pH LAPNB_MAX_OUTER_FOLDS=1 LAPNB_SKIP_FULL_67K=1`. With a fold limit, nothing is saved.

## 0. Settings and imports

In [ ]:
import os
import pickle
import time
from collections import defaultdict
from math import asin, cos, radians, sin, sqrt
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from scipy import sparse
from scipy.sparse.linalg import splu
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

BASE_DIR = Path.cwd()
FULL_GRAPH_FILE = BASE_DIR / "graphs/colorado/largest_river_component.pkl"
MASTER_GRAPH_FILE = BASE_DIR / "graphs/colorado/master_compressed_river_graph.pkl"
HOURLY_FILE = BASE_DIR / "data/usgs/usgs_all_stations_hourly.csv"
MERIT_FILE = BASE_DIR / "data/colorado/merit_colorado_float_native.tif"
OUT_DIR = BASE_DIR / "results/laplacian_fixed"

# ---------------- run settings ----------------
RUN_FULL_67K = os.environ.get("LAPNB_SKIP_FULL_67K") != "1"      # the slow 67,401-node fits in §6
MAX_OUTER_FOLDS = int(os.environ.get("LAPNB_MAX_OUTER_FOLDS", "0")) or None   # smoke tests only
ALL_TARGETS = ["DO concentration", "Water temperature", "pH", "Specific conductance", "Turbidity"]
TARGETS = os.environ["LAPNB_TARGETS"].split(",") if os.environ.get("LAPNB_TARGETS") else ALL_TARGETS
SAVE_RESULTS = MAX_OUTER_FOLDS is None
if SAVE_RESULTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- model settings (same as the original notebook) ----------------
WQ_VARIABLES = ["DO concentration", "Water temperature", "Specific conductance", "Turbidity", "pH"]
TARGET_FEATURES = {
    "DO concentration": ["Water temperature", "Specific conductance"],
    "Water temperature": ["Specific conductance"],
    "pH": ["Specific conductance"],
    "Specific conductance": ["Water temperature"],
    "Turbidity": ["Specific conductance", "pH"],
}
LINEAR_ALPHA_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
GRAPH_LAMBDA_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
GRAPH_BETA_GRID = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

print("Targets:", TARGETS)
print("Full 67k fits:", RUN_FULL_67K, "| max outer folds:", MAX_OUTER_FOLDS, "| save results:", SAVE_RESULTS)

## 1. Load graphs and hourly data

Same station → node mapping as the original notebook (cell 1). Fourier terms up to K = 5 are added here
(`doy` includes the fractional day, as in the original's Fourier section).

In [ ]:
with open(FULL_GRAPH_FILE, "rb") as f:
    G_full = pickle.load(f)
with open(MASTER_GRAPH_FILE, "rb") as f:
    G_master = pickle.load(f)

station_to_node = {}
for node, data in G_master.nodes(data=True):
    for cluster in data.get("usgs_clusters", []):
        stations = cluster.get("member_stations", [])
        if not stations and cluster.get("anchor_station") is not None:
            stations = [cluster["anchor_station"]]
        for station in stations:
            station_to_node[station] = node

wq_nodes = sorted(
    node for node, data in G_master.nodes(data=True)
    if data.get("usgs_station_count", 0) > 0 or len(data.get("usgs_clusters", [])) > 0
)
N_WQ = len(wq_nodes)

hourly = pd.read_csv(HOURLY_FILE)
hourly["time"] = pd.to_datetime(hourly["time"], utc=True, errors="coerce")
hourly["node"] = hourly["station_id"].map(station_to_node)
hourly = hourly[hourly["node"].notna()].copy()
hourly["node"] = hourly["node"].astype(int)

node_hourly = (
    hourly[["node", "time"] + WQ_VARIABLES]
    .groupby(["node", "time"], as_index=False)[WQ_VARIABLES]
    .median()
)

MAX_K = 5
hour = node_hourly["time"].dt.hour + node_hourly["time"].dt.minute / 60.0
doy = node_hourly["time"].dt.dayofyear + hour / 24.0
for k in range(1, MAX_K + 1):
    node_hourly[f"hour_sin_{k}"] = np.sin(2 * np.pi * k * hour / 24.0)
    node_hourly[f"hour_cos_{k}"] = np.cos(2 * np.pi * k * hour / 24.0)
    node_hourly[f"doy_sin_{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
    node_hourly[f"doy_cos_{k}"] = np.cos(2 * np.pi * k * doy / 365.25)


def fourier_features(K):
    out = []
    for k in range(1, K + 1):
        out += [f"hour_sin_{k}", f"hour_cos_{k}", f"doy_sin_{k}", f"doy_cos_{k}"]
    return out


print("Full graph:", G_full.number_of_nodes(), "nodes |", "compressed:", G_master.number_of_nodes(), "nodes")
print("WQ nodes:", N_WQ, "| node-hours:", len(node_hourly))

## 2. MERIT node attributes and static predictors

Bands 1–4 of `merit_colorado_float_native.tif` are upstream area, elevation, river width and HAND.

**Fix 4.** HAND (height above nearest drainage) is 0 on river-channel pixels, and every WQ node is a river
pixel. The check below shows that, so HAND is left out of the static predictors.

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = (radians(float(v)) for v in (lon1, lat1, lon2, lat2))
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return 2.0 * 6371000.0 * asin(sqrt(a))


def edge_length_m(G, u, v):
    length_m = G.edges[u, v].get("length_m")
    if length_m is not None and np.isfinite(float(length_m)) and float(length_m) > 0:
        return float(length_m)
    a, b = G.nodes[u], G.nodes[v]
    return haversine_m(a["longitude"], a["latitude"], b["longitude"], b["latitude"])


full_node_order = list(G_full.nodes())
full_index = {n: i for i, n in enumerate(full_node_order)}
coords = [(float(G_full.nodes[n]["longitude"]), float(G_full.nodes[n]["latitude"])) for n in full_node_order]
with rasterio.open(MERIT_FILE) as src:
    samples = np.asarray(list(src.sample(coords, indexes=[1, 2, 3, 4])), dtype=float)
upa, elevation, width, hand = samples.T

# downstream slope from the first successor (as in the original cell 100)
local_slope = np.full(len(full_node_order), np.nan)
for i, node in enumerate(full_node_order):
    succ = list(G_full.successors(node))
    if succ:
        j = full_index[succ[0]]
        d = edge_length_m(G_full, node, succ[0])
        if d > 0 and np.isfinite(elevation[i]) and np.isfinite(elevation[j]):
            local_slope[i] = max((elevation[i] - elevation[j]) / d, 0.0)

static_node_df = pd.DataFrame({
    "node": full_node_order,
    "elevation_m": elevation,
    "log_upa": np.log1p(np.clip(upa, 0, None)),
    "log_width": np.log1p(np.clip(width, 0, None)),
    "hand_m": hand,
    "local_slope": local_slope,
})

print("HAND at the WQ nodes:")
print(static_node_df.set_index("node").loc[wq_nodes, "hand_m"].describe().to_string())

STATIC_FEATURES = ["elevation_m", "log_upa", "log_width", "local_slope"]   # HAND dropped (fix 4)
print("\nStatic predictors used:", STATIC_FEATURES)
print("Missing values at WQ nodes:",
      int(static_node_df.set_index("node").loc[wq_nodes, STATIC_FEATURES].isna().sum().sum()))

## 3. Edge features and the five weighting schemes

Edge weight `w = exp(score) / distance_km`, where `score` mixes z-scored log area, log width and log slope
(z clipped to ±4, score clipped to ±5, as in the original cells 18–20).

**Fixes 3 and 6.** The original also had a `log_discharge` feature, with discharge predicted as
`exp(a + b·log UPA)`. Then `log √(Q_u Q_v) = a + b · log √(A_u A_v)`, so `z_log_discharge` is identical to
`z_log_area`. Any "discharge" scheme just doubled the area weight, and fitting `a, b` used discharge from nodes that
are later held out. Discharge is dropped. The five schemes kept are the ones in the original's
`effective_laplacians_clean`.

In [ ]:
edge_rows = []
for u, v in G_full.edges():          # directed u -> v (downstream)
    iu, iv = full_index[u], full_index[v]
    d_m = edge_length_m(G_full, u, v)
    Au, Av = max(upa[iu], 1e-8), max(upa[iv], 1e-8)
    Wu, Wv = max(width[iu], 1e-8), max(width[iv], 1e-8)
    drop = max(elevation[iu] - elevation[iv], 0.0)
    edge_rows.append({
        "u": u, "v": v,
        "distance_km": d_m / 1000.0,
        "log_area": np.log(np.sqrt(Au * Av)),
        "log_width": np.log(np.sqrt(Wu * Wv)),
        "log_slope": np.log1p(drop / max(d_m, 1e-8)),
    })
edge_features = pd.DataFrame(edge_rows)

for col in ["log_area", "log_width", "log_slope"]:
    vals = edge_features[col].to_numpy(float)
    vals = np.where(np.isfinite(vals), vals, np.median(vals[np.isfinite(vals)]))
    std = vals.std(ddof=1)
    std = std if np.isfinite(std) and std > 1e-12 else 1.0
    edge_features["z_" + col] = np.clip((vals - vals.mean()) / std, -4.0, 4.0)

WEIGHT_SCHEMES = {
    "distance_only": {"area": 0.0, "width": 0.0, "slope": 0.0},
    "area": {"area": 0.5, "width": 0.0, "slope": 0.0},
    "width": {"area": 0.0, "width": 0.5, "slope": 0.0},
    "slope": {"area": 0.0, "width": 0.0, "slope": -0.5},
    "area_width": {"area": 0.35, "width": 0.35, "slope": 0.0},
}


def scheme_edge_weights(scheme):
    score = (scheme["area"] * edge_features["z_log_area"]
             + scheme["width"] * edge_features["z_log_width"]
             + scheme["slope"] * edge_features["z_log_slope"]).to_numpy()
    w = np.exp(np.clip(score, -5.0, 5.0)) / np.maximum(edge_features["distance_km"].to_numpy(), 1e-6)
    assert np.all(np.isfinite(w)) and np.all(w > 0)
    return w


print(edge_features[["distance_km", "z_log_area", "z_log_width", "z_log_slope"]].describe().to_string())

## 4. Laplacians: full, compressed and Kron-reduced

* **Full:** `L = D − W` on the 67,401-node graph (undirected).
* **Compressed:** each edge of the 67-node graph is a chain of full-graph edges (`original_path`). Its weight is
  the series conductance `1 / Σ (1/w)` (original cell 40).
* **Kron:** `L_BB − L_BI L_II⁻¹ L_IB` onto the 26 WQ nodes, symmetrised with exact zero row sums, then divided
  by its mean positive diagonal (the `scale`), as in the original.

In [ ]:
def laplacian_from_edges(n, i, j, w):
    W = sparse.coo_matrix((np.r_[w, w], (np.r_[i, j], np.r_[j, i])), shape=(n, n)).tocsr()
    return (sparse.diags(np.asarray(W.sum(axis=1)).ravel()) - W).tocsc()


def kron_reduce(L, b_idx):
    n = L.shape[0]
    interior = np.setdiff1d(np.arange(n), b_idx)
    Lr = L.tocsr()
    L_BB = Lr[b_idx][:, b_idx].toarray()
    L_BI = Lr[b_idx][:, interior].tocsc()
    L_IB = Lr[interior][:, b_idx].toarray()
    lu_II = splu(Lr[interior][:, interior].tocsc())
    raw = np.asarray(L_BB - L_BI @ lu_II.solve(L_IB))
    clean = (raw + raw.T) / 2.0
    np.fill_diagonal(clean, 0.0)
    np.fill_diagonal(clean, -clean.sum(axis=1))
    return clean, raw, dict(interior=interior, L_BB=L_BB, L_IB=L_IB, lu_II=lu_II)


def kron_scale(Lk):
    d = np.diag(Lk)
    return d[d > 1e-12].mean()


edge_i = np.array([full_index[u] for u in edge_features["u"]])
edge_j = np.array([full_index[v] for v in edge_features["v"]])

comp_node_order = list(G_master.nodes())
comp_index = {n: i for i, n in enumerate(comp_node_order)}
comp_edges = list(G_master.edges(data=True))
b_full = np.array([full_index[n] for n in wq_nodes])
b_comp = np.array([comp_index[n] for n in wq_nodes])

LAP = {}   # LAP[(graph, scheme)] = dict(L, L_kron, L_kron_raw, parts, scale, index, b_idx)
for scheme_name, scheme in WEIGHT_SCHEMES.items():
    w_full = scheme_edge_weights(scheme)
    L_full = laplacian_from_edges(len(full_node_order), edge_i, edge_j, w_full)

    # compressed: series conductance along each original path
    w_lookup = {frozenset((u, v)): w for u, v, w in zip(edge_features["u"], edge_features["v"], w_full)}
    ci, cj, cw = [], [], []
    for u, v, data in comp_edges:
        path = data.get("original_path") or [u, v]
        resistance = sum(1.0 / w_lookup[frozenset((a, b))] for a, b in zip(path[:-1], path[1:]))
        ci.append(comp_index[u]); cj.append(comp_index[v]); cw.append(1.0 / resistance)
    L_comp = laplacian_from_edges(len(comp_node_order), np.array(ci), np.array(cj), np.array(cw))

    for graph, L, index, b_idx in (("uncompressed", L_full, full_index, b_full),
                                   ("compressed", L_comp, comp_index, b_comp)):
        Lk, Lk_raw, parts = kron_reduce(L, b_idx)
        LAP[(graph, scheme_name)] = dict(L=L, L_kron=Lk, L_kron_raw=Lk_raw, parts=parts,
                                         scale=kron_scale(Lk), index=index, b_idx=b_idx)
    print(f"{scheme_name:<14} full {L_full.shape[0]}x{L_full.shape[0]}  compressed "
          f"{L_comp.shape[0]}x{L_comp.shape[0]}  Kron 26x26  "
          f"scale(uncompressed)={LAP[('uncompressed', scheme_name)]['scale']:.5g}")

## 5. How different are the Laplacians?

The Kron and full matrices are different sizes, so they can't be subtracted directly. Instead we check what Kron
reduction must preserve exactly:

* **Effective resistance:** the resistance between every pair of WQ nodes, computed on the full Laplacian
  (by grounding one WQ node and doing sparse solves) and on the Kron matrix.
* **Dirichlet energy:** for random WQ values `x_B` with the harmonic extension `x` into the rest of the graph,
  `xᵀLx` on the full graph should equal `x_Bᵀ L_kron x_B`.
* **Kron(uncompressed) vs Kron(compressed):** a direct matrix difference.
* **Reference:** the naive sub-matrix `L_BB` (just deleting the other nodes) shows what a *wrong* reduction
  looks like.

In [ ]:
def rel_fro(A, B):
    return np.linalg.norm(A - B) / np.linalg.norm(A)


def resistance_from_gram(G):
    d = np.diag(G)
    return d[:, None] + d[None, :] - 2.0 * G


def effective_resistance_full(L, b_idx):
    keep = np.setdiff1d(np.arange(L.shape[0]), [b_idx[0]])
    pos = {g: k for k, g in enumerate(keep)}
    lu = splu(L.tocsr()[keep][:, keep].tocsc())
    rhs = np.zeros((len(keep), len(b_idx) - 1))
    for c, b in enumerate(b_idx[1:]):
        rhs[pos[b], c] = 1.0
    sol = lu.solve(rhs)
    G = np.zeros((len(b_idx), len(b_idx)))
    G[1:, 1:] = sol[[pos[b] for b in b_idx[1:]], :]
    return resistance_from_gram(G)


def effective_resistance_small(Lk):
    G = np.zeros_like(Lk)
    G[1:, 1:] = np.linalg.inv(Lk[1:, 1:])
    return resistance_from_gram(G)


def energy_error(L, b_idx, Lk, parts, trials=5, seed=0):
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(trials):
        xB = rng.standard_normal(len(b_idx))
        x = np.zeros(L.shape[0])
        x[b_idx] = xB
        x[parts["interior"]] = -parts["lu_II"].solve(parts["L_IB"] @ xB)
        e_full, e_kron = float(x @ (L @ x)), float(xB @ Lk @ xB)
        worst = max(worst, abs(e_full - e_kron) / abs(e_full))
    return worst


off_diag = ~np.eye(N_WQ, dtype=bool)
check_rows = []
for scheme_name in WEIGHT_SCHEMES:
    R = {}
    for graph in ("uncompressed", "compressed"):
        lap = LAP[(graph, scheme_name)]
        R_full = effective_resistance_full(lap["L"], lap["b_idx"])
        R_kron = effective_resistance_small(lap["L_kron"])
        R[graph] = R_full
        check_rows.append({
            "scheme": scheme_name, "graph": graph,
            "eff. resistance max rel err (Kron vs full)":
                (np.abs(R_full - R_kron)[off_diag] / R_full[off_diag]).max(),
            "energy max rel err (Kron vs full)": energy_error(lap["L"], lap["b_idx"], lap["L_kron"], lap["parts"]),
            "Kron clean-up change (rel fro)": rel_fro(lap["L_kron_raw"], lap["L_kron"]),
            "naive L_BB vs Kron (rel fro, should be large)": rel_fro(lap["L_kron"], lap["parts"]["L_BB"]),
        })
    ku, kc = LAP[("uncompressed", scheme_name)], LAP[("compressed", scheme_name)]
    check_rows.append({
        "scheme": scheme_name, "graph": "uncompressed vs compressed",
        "Kron scaled matrices (rel fro)": rel_fro(ku["L_kron"] / ku["scale"], kc["L_kron"] / kc["scale"]),
        "Kron max abs diff (unscaled)": np.abs(ku["L_kron"] - kc["L_kron"]).max(),
        "eff. resistance max rel diff (full graphs)":
            (np.abs(R["uncompressed"] - R["compressed"])[off_diag] / R["uncompressed"][off_diag]).max(),
    })

matrix_checks = pd.DataFrame(check_rows)
display(matrix_checks.style.format("{:.2e}", subset=matrix_checks.columns[2:], na_rep=""))
if SAVE_RESULTS:
    matrix_checks.to_csv(OUT_DIR / "kron_vs_full_matrix_checks.csv", index=False)

## 6. Kron vs full: does R² change?

### Model
For node *i* and hour *t*: `ŷ_it = β₀ + z_itᵀβ + g_i`, fitted by minimising

```
Σ_{i∈train} (1/n_i) Σ_t (y_it − β₀ − z_itᵀβ − g_i)²  +  λ_β Σ_{k≥1} β_k²  +  λ gᵀ L̃ g     s.t.  Σ_{WQ nodes} g_i = 0
```

with `L̃` either the scaled Kron matrix (g on the 26 WQ nodes) or the scaled full Laplacian (g on every node).
The solver eliminates g by block elimination of the **full** system. One factorisation of
`diag(mask) + λL̃` is shared across the λ_β grid. All four variants use exactly the same code; only the Laplacian
differs.

Every fit is computed from per-node moments, as in the original notebook, so it is exact and fast.

In [ ]:
# ---------------- per-node moments ----------------
def make_node_stats(df, target, features):
    stats = {}
    for node, group in df.groupby("node"):
        X = group[features].to_numpy(float)
        y = group[target].to_numpy(float)
        n = len(group)
        stats[int(node)] = dict(n=n, mean_x=X.mean(axis=0), exx=X.T @ X / n, exy=X.T @ y / n,
                                mean_y=y.mean(), mean_y2=np.mean(y ** 2))
    return stats


def slice_stats(stats, idx):
    idx = np.asarray(idx)
    return {n: dict(s, mean_x=s["mean_x"][idx], exx=s["exx"][np.ix_(idx, idx)], exy=s["exy"][idx])
            for n, s in stats.items()}


def standard_moments(stat, mean_x, std_x):
    mx, exx, exy, my = stat["mean_x"], stat["exx"], stat["exy"], stat["mean_y"]
    zmean = (mx - mean_x) / std_x
    zexx = (exx - np.outer(mx, mean_x) - np.outer(mean_x, mx) + np.outer(mean_x, mean_x)) / np.outer(std_x, std_x)
    zexy = (exy - mean_x * my) / std_x
    p = len(mean_x)
    q2 = np.empty((p + 1, p + 1))
    q2[0, 0] = 1.0
    q2[0, 1:] = q2[1:, 0] = zmean
    q2[1:, 1:] = zexx
    return np.concatenate(([1.0], zmean)), q2, np.concatenate(([my], zexy))


def train_moments(stats, train_nodes):
    """Everything about a training set that does not depend on the hyper-parameters."""
    mean_x = np.mean([stats[n]["mean_x"] for n in train_nodes], axis=0)
    ex2 = np.mean([np.diag(stats[n]["exx"]) for n in train_nodes], axis=0)
    std_x = np.sqrt(np.maximum(ex2 - mean_x ** 2, 1e-12))
    q = len(mean_x) + 1
    A11, rb, qmean = np.zeros((q, q)), np.zeros(q), {}
    for n in train_nodes:
        qm, q2, qy = standard_moments(stats[n], mean_x, std_x)
        A11 += q2
        rb += qy
        qmean[n] = qm
    return dict(mean_x=mean_x, std_x=std_x, q=q, A11=A11, rb=rb, qmean=qmean,
                mean_y={n: stats[n]["mean_y"] for n in train_nodes})


def val_moments(stat, tm):
    _, q2, qy = standard_moments(stat, tm["mean_x"], tm["std_x"])
    return q2, qy, stat["mean_y2"]


def mse_from(vm, coef):
    q2, qy, my2 = vm
    return max(float(my2 - 2.0 * coef @ qy + coef @ q2 @ coef), 0.0)


def ridge_matrix(q):
    R = np.eye(q)
    R[0, 0] = 0.0
    return R


# ---------------- linear model ----------------
def fit_linear(tm, alpha):
    beta = np.linalg.solve(tm["A11"] + alpha * ridge_matrix(tm["q"]), tm["rb"])
    return dict(beta=beta, g=None, mean_x=tm["mean_x"], std_x=tm["std_x"])


# ---------------- graph model ----------------
class GraphSpace:
    """A scaled Laplacian plus where the WQ nodes sit inside it."""

    def __init__(self, L_scaled, node_pos, constraint_idx):
        self.L = sparse.csc_matrix(L_scaled)
        self.N = self.L.shape[0]
        self.dense = self.N <= 2000
        self.L_dense = self.L.toarray() if self.dense else None
        self.node_pos = node_pos                  # WQ node id -> row of L
        self.c = np.zeros(self.N)                 # gauge: sum of g over the WQ nodes = 0
        self.c[constraint_idx] = 1.0


def prepare_graph(tm, lam, space):
    """Block-eliminate g from the full system
         [A11+λβR  B   0] [β ]   [r_β]
         [Bᵀ       K   c] [g ] = [r_g]      K = diag(mask) + λ L̃
         [0        cᵀ  0] [μ ]   [0  ]
    using one factorisation of K, shared across the λ_β grid."""
    N, q = space.N, tm["q"]
    Bt, rg, mask = np.zeros((N, q)), np.zeros(N), np.zeros(N)
    for n, qm in tm["qmean"].items():
        j = space.node_pos[n]
        Bt[j] += qm
        mask[j] = 1.0
        rg[j] = tm["mean_y"][n]
    rhs = np.column_stack((Bt, rg, space.c))
    if space.dense:
        sol = np.linalg.solve(np.diag(mask) + lam * space.L_dense, rhs)
    else:
        sol = splu((sparse.diags(mask) + lam * space.L).tocsc(), permc_spec="MMD_AT_PLUS_A").solve(rhs)
    X, u, v = sol[:, :q], sol[:, q], sol[:, q + 1]
    c = space.c
    return dict(tm=tm, X=X, u=u, v=v, BtX=Bt.T @ X, Btu=Bt.T @ u, Btv=Bt.T @ v,
                cX=c @ X, cu=c @ u, cv=c @ v)


def solve_graph(prep, lambda_beta):
    tm, q = prep["tm"], prep["tm"]["q"]
    S = np.empty((q + 1, q + 1))
    S[:q, :q] = tm["A11"] + lambda_beta * ridge_matrix(q) - prep["BtX"]
    S[:q, q] = -prep["Btv"]
    S[q, :q] = prep["cX"]
    S[q, q] = prep["cv"]
    z = np.linalg.solve(S, np.concatenate((tm["rb"] - prep["Btu"], [prep["cu"]])))
    beta, mu = z[:q], z[q]
    g = prep["u"] - prep["X"] @ beta - prep["v"] * mu
    return dict(beta=beta, g=g, mean_x=tm["mean_x"], std_x=tm["std_x"])


def graph_coef(model, node, space):
    coef = model["beta"].copy()
    coef[0] += model["g"][space.node_pos[node]]
    return coef


def predict_rows(df, features, model, space=None):
    Z = (df[features].to_numpy(float) - model["mean_x"]) / model["std_x"]
    pred = np.column_stack((np.ones(len(df)), Z)) @ model["beta"]
    if model["g"] is not None:
        pred += model["g"][[space.node_pos[int(n)] for n in df["node"]]]
    return pred


# ---------------- inner-LOO tuning ----------------
def tune_linear(stats, outer_train, alpha_grid=LINEAR_ALPHA_GRID):
    totals = dict.fromkeys(alpha_grid, 0.0)
    for v in outer_train:
        tm = train_moments(stats, [n for n in outer_train if n != v])
        vm = val_moments(stats[v], tm)
        for a in alpha_grid:
            totals[a] += mse_from(vm, fit_linear(tm, a)["beta"])
    best = min(totals, key=totals.get)
    return best, totals[best] / len(outer_train)


def tune_graph(stats, outer_train, spaces):
    """spaces = {scheme: GraphSpace}. Returns the mean inner-LOO MSE for every (scheme, λ, λ_β)."""
    totals = defaultdict(float)
    for v in outer_train:
        tm = train_moments(stats, [n for n in outer_train if n != v])
        vm = val_moments(stats[v], tm)
        for scheme, space in spaces.items():
            for lam in GRAPH_LAMBDA_GRID:
                prep = prepare_graph(tm, lam, space)
                for lb in GRAPH_BETA_GRID:
                    totals[(scheme, lam, lb)] += mse_from(vm, graph_coef(solve_graph(prep, lb), v, space))
    return {k: t / len(outer_train) for k, t in totals.items()}


def best_key(scores, schemes=None):
    keys = [k for k in scores if schemes is None or k[0] in schemes]
    return min(keys, key=scores.get)


def weighted_r2_rmse(y, pred, nodes):
    w = pd.Series(nodes).map(1.0 / pd.Series(nodes).value_counts()).to_numpy()
    return r2_score(y, pred, sample_weight=w), np.sqrt(mean_squared_error(y, pred, sample_weight=w))

In [ ]:
# ---------------- the four Laplacian variants, each holding all five schemes ----------------
kron_pos = {n: i for i, n in enumerate(wq_nodes)}
SCHEMES = list(WEIGHT_SCHEMES)


def make_spaces(variant):
    graph = "uncompressed" if variant.endswith("_uncompressed") else "compressed"
    out = {}
    for s in SCHEMES:
        lap = LAP[(graph, s)]
        if variant.startswith("kron"):
            out[s] = GraphSpace(lap["L_kron"] / lap["scale"], kron_pos, np.arange(N_WQ))
        else:   # full Laplacian, divided by the SAME scale as its Kron reduction
            out[s] = GraphSpace(lap["L"] / lap["scale"], {n: lap["index"][n] for n in wq_nodes}, lap["b_idx"])
    return out


VARIANTS = ["kron_uncompressed", "kron_compressed", "full_compressed"] + (["full_uncompressed"] if RUN_FULL_67K else [])
VARIANT_SPACES = {v: make_spaces(v) for v in VARIANTS}
print("Variants:", VARIANTS)

Each outer fold tunes (scheme, λ, λ_β) once per variant over all five schemes. Two results come out of the same
tuning scores:

* **`distance_only`**: best (λ, λ_β) with the 1/distance Laplacian only. This matches the original cells 1–13 and
  `results/full_graph_nested_loo_results.csv`.
* **`scheme_tuned`**: best (scheme, λ, λ_β) over all five. This matches the original cells 24–26 and 35, and the
  `Linear + Laplacian` rows in `results/all_models_node_loo_results.csv`.

In [ ]:
def nested_loo_kron_check(df, target, features, variant_spaces, max_outer=None):
    stats = make_node_stats(df, target, features)
    nodes = sorted(stats)
    outer = nodes[:max_outer] if max_outer else nodes
    parts, tuning = [], []
    for fold, test_node in enumerate(outer, 1):
        train = [n for n in nodes if n != test_node]
        test_df = df[df["node"] == test_node]
        tm = train_moments(stats, train)
        out = pd.DataFrame({"node": test_df["node"].to_numpy(), "y_true": test_df[target].to_numpy()})
        alpha, _ = tune_linear(stats, train)
        out["linear"] = predict_rows(test_df, features, fit_linear(tm, alpha))
        trow = {"target": target, "heldout_node": test_node, "linear_alpha": alpha}
        msg = []
        for vname, spaces in variant_spaces.items():
            t0 = time.time()
            scores = tune_graph(stats, train, spaces)
            for mode, allowed in (("distance_only", ["distance_only"]), ("scheme_tuned", None)):
                scheme, lam, lb = best_key(scores, allowed)
                model = solve_graph(prepare_graph(tm, lam, spaces[scheme]), lb)
                out[f"{mode}|{vname}"] = predict_rows(test_df, features, model, spaces[scheme])
                trow[f"{mode}|{vname}"] = (scheme, lam, lb)
            msg.append(f"{vname} {time.time() - t0:.1f}s")
        print(f"  {target:<21} fold {fold:>2}/{len(outer)}  " + " | ".join(msg), flush=True)
        parts.append(out)
        tuning.append(trow)

    preds = pd.concat(parts, ignore_index=True)
    rows = []
    for col in [c for c in preds if c not in ("node", "y_true")]:
        r2, rmse = weighted_r2_rmse(preds["y_true"].to_numpy(), preds[col].to_numpy(), preds["node"].to_numpy())
        mode, variant = col.split("|") if "|" in col else ("linear", "linear")
        rows.append({"Target": target, "Nodes": len(nodes), "Mode": mode, "Variant": variant, "R2": r2, "RMSE": rmse})
    return pd.DataFrame(rows), pd.DataFrame(tuning)


kron_check_results, kron_check_tuning = [], []
t_start = time.time()
for target in TARGETS:
    features = TARGET_FEATURES[target]
    df = node_hourly[["node", "time", target] + features].dropna().copy()
    res, tun = nested_loo_kron_check(df, target, features, VARIANT_SPACES, MAX_OUTER_FOLDS)
    kron_check_results.append(res)
    kron_check_tuning.append(tun)
kron_check_results = pd.concat(kron_check_results, ignore_index=True)
kron_check_tuning = pd.concat(kron_check_tuning, ignore_index=True)
print(f"\nDone in {time.time() - t_start:.0f}s")

In [ ]:
r2_table = kron_check_results[kron_check_results["Mode"] != "linear"].pivot_table(
    index=["Target", "Mode"], columns="Variant", values="R2").reindex(columns=VARIANTS)
for v in VARIANTS[1:]:
    r2_table[f"{v} − kron_uncompressed"] = r2_table[v] - r2_table["kron_uncompressed"]

# how often each variant picked the same (scheme, λ, λ_β) as kron_uncompressed
agree_rows = []
for target, t in kron_check_tuning.groupby("target"):
    for mode in ("distance_only", "scheme_tuned"):
        row = {"Target": target, "Mode": mode}
        for v in VARIANTS[1:]:
            row[f"{v} same hyper-params"] = (t[f"{mode}|{v}"] == t[f"{mode}|kron_uncompressed"]).mean()
        agree_rows.append(row)
agree = pd.DataFrame(agree_rows).set_index(["Target", "Mode"])

print("R² by Laplacian variant")
display(r2_table.style.format("{:.6f}", subset=VARIANTS).format("{:.2e}", subset=[c for c in r2_table if "−" in c]))
print("Fraction of outer folds choosing the same hyper-parameters as kron_uncompressed")
display(agree.style.format("{:.0%}"))

**Reproduction check against the original notebook's saved results.** `kron_uncompressed` should reproduce them:
`distance_only` against `results/full_graph_nested_loo_results.csv`, and `scheme_tuned` against the
`Linear + Laplacian` rows of `results/all_models_node_loo_results.csv`. (Full runs only.)

In [ ]:
if MAX_OUTER_FOLDS is None:
    prev_dist = pd.read_csv(BASE_DIR / "results/full_graph_nested_loo_results.csv").set_index("Target")["Graph R2"]
    prev_all = pd.read_csv(BASE_DIR / "results/all_models_node_loo_results.csv")
    prev_scheme = prev_all[prev_all["Model"] == "Linear + Laplacian"].set_index("Target")["R2"]
    repro = pd.DataFrame({
        "distance_only (this run)": r2_table.xs("distance_only", level="Mode")["kron_uncompressed"],
        "distance_only (saved)": prev_dist,
        "scheme_tuned (this run)": r2_table.xs("scheme_tuned", level="Mode")["kron_uncompressed"],
        "scheme_tuned (saved)": prev_scheme,
    }).loc[TARGETS]
    repro["max |diff|"] = np.maximum((repro.iloc[:, 0] - repro.iloc[:, 1]).abs(),
                                     (repro.iloc[:, 2] - repro.iloc[:, 3]).abs())
    display(repro.style.format("{:.6f}").format("{:.1e}", subset=["max |diff|"]))
else:
    print("Skipped: fold limit is set.")

if SAVE_RESULTS:
    kron_check_results.to_csv(OUT_DIR / "kron_vs_full_r2.csv", index=False)
    kron_check_tuning.astype(str).to_csv(OUT_DIR / "kron_vs_full_tuning.csv", index=False)

## 7. pH target transforms, with the scaled Laplacian (fix 1)

The original cells 63–67 compared fitting pH raw, z-scored, or as H⁺ = 10^(7−pH). They used
`colorado_effective_laplacians["Full"]`, which was **not** divided by the Kron scale (≈ 0.048). Every other
section divides by it, so the same λ there was ~21× weaker. That section also used a 1e-10 ridge instead of the
sum-to-zero gauge (this doesn't change predictions).

Here the same experiment is run with the **scaled** `distance_only` Kron Laplacian (the fix), and with the
**unscaled** one alongside it to show how much the bug mattered. As in the original:

* predictors are specific conductance plus K = 1 Fourier terms;
* the linear model's ridge grid is `GRAPH_BETA_GRID`;
* transform parameters are fitted on the training nodes only;
* validation error is measured on the pH scale.

In [ ]:
PH_FEATURES = ["Specific conductance"] + fourier_features(1)
ph_df = node_hourly[["node", "time", "pH"] + PH_FEATURES].dropna().copy()
ph_df["hplus"] = np.power(10.0, 7.0 - ph_df["pH"])
ph_nodes = sorted(ph_df["node"].unique())
ph_rows = {n: (g[PH_FEATURES].to_numpy(float), g["pH"].to_numpy(float)) for n, g in ph_df.groupby("node")}
ph_stats_raw = make_node_stats(ph_df, "pH", PH_FEATURES)
ph_stats_hplus = make_node_stats(ph_df, "hplus", PH_FEATURES)

lap_d = LAP[("uncompressed", "distance_only")]
PH_SPACES = {
    "scaled (fixed)": GraphSpace(lap_d["L_kron"] / lap_d["scale"], kron_pos, np.arange(N_WQ)),
    "unscaled (original)": GraphSpace(lap_d["L_kron"], kron_pos, np.arange(N_WQ)),
}


def ph_stats_for(mode, train_nodes):
    """Node moments of the transformed target + inverse transform (parameters from training nodes only)."""
    if mode == "raw":
        return ph_stats_raw, (lambda y: (y, 0))
    if mode == "zscore":
        mu = np.mean([ph_stats_raw[n]["mean_y"] for n in train_nodes])
        var = np.mean([ph_stats_raw[n]["mean_y2"] for n in train_nodes]) - mu ** 2
        sd = np.sqrt(var) if var > 1e-24 else 1.0
        stats = {n: dict(s, mean_y=(s["mean_y"] - mu) / sd,
                         mean_y2=(s["mean_y2"] - 2 * mu * s["mean_y"] + mu ** 2) / sd ** 2,
                         exy=(s["exy"] - s["mean_x"] * mu) / sd) for n, s in ph_stats_raw.items()}
        return stats, (lambda y: (y * sd + mu, 0))
    if mode == "hplus":
        return ph_stats_hplus, (lambda y: (7.0 - np.log10(np.maximum(y, 1e-8)), int(np.sum(y <= 0))))
    raise ValueError(mode)


def ph_predict(node, model, inverse, space=None):
    X, _ = ph_rows[node]
    Z = (X - model["mean_x"]) / model["std_x"]
    coef = model["beta"] if space is None else graph_coef(model, node, space)
    return inverse(np.column_stack((np.ones(len(X)), Z)) @ coef)


def ph_nested_loo(mode, max_outer=None):
    outer = ph_nodes[:max_outer] if max_outer else ph_nodes
    preds, clips, tuning = [], 0, []
    for test in outer:
        train = [n for n in ph_nodes if n != test]
        lin_tot = defaultdict(float)
        gr_tot = {name: defaultdict(float) for name in PH_SPACES}
        for v in train:
            inner = [n for n in train if n != v]
            stats, inv = ph_stats_for(mode, inner)
            tm = train_moments(stats, inner)
            y_val = ph_rows[v][1]
            for a in GRAPH_BETA_GRID:
                lin_tot[a] += np.mean((y_val - ph_predict(v, fit_linear(tm, a), inv)[0]) ** 2)
            for name, space in PH_SPACES.items():
                for lam in GRAPH_LAMBDA_GRID:
                    prep = prepare_graph(tm, lam, space)
                    for lb in GRAPH_BETA_GRID:
                        p = ph_predict(v, solve_graph(prep, lb), inv, space)[0]
                        gr_tot[name][(lam, lb)] += np.mean((y_val - p) ** 2)
        stats, inv = ph_stats_for(mode, train)
        tm = train_moments(stats, train)
        y_test = ph_rows[test][1]
        out = pd.DataFrame({"node": test, "y_true": y_test})
        a = min(lin_tot, key=lin_tot.get)
        out["linear"], c = ph_predict(test, fit_linear(tm, a), inv)
        clips += c
        trow = {"mode": mode, "heldout_node": test, "linear_alpha": a}
        for name, space in PH_SPACES.items():
            lam, lb = min(gr_tot[name], key=gr_tot[name].get)
            out[name], c = ph_predict(test, solve_graph(prepare_graph(tm, lam, space), lb), inv, space)
            clips += c
            trow[name] = (lam, lb)
        preds.append(out)
        tuning.append(trow)
    preds = pd.concat(preds, ignore_index=True)
    row = {"Transformation": mode, "Nodes": len(ph_nodes), "H+ clipped predictions": clips}
    for col in ["linear"] + list(PH_SPACES):
        row[f"{col} R2"], row[f"{col} RMSE"] = weighted_r2_rmse(
            preds["y_true"].to_numpy(), preds[col].to_numpy(), preds["node"].to_numpy())
    return row, pd.DataFrame(tuning)


if "pH" in TARGETS:
    t0 = time.time()
    ph_out = [ph_nested_loo(m, MAX_OUTER_FOLDS) for m in ("raw", "zscore", "hplus")]
    ph_results = pd.DataFrame([r for r, _ in ph_out])
    ph_tuning = pd.concat([t for _, t in ph_out], ignore_index=True)
    ph_results["graph R2 change from fix"] = ph_results["scaled (fixed) R2"] - ph_results["unscaled (original) R2"]
    print(f"pH transforms done in {time.time() - t0:.0f}s")
    display(ph_results.style.format("{:.4f}", subset=[c for c in ph_results if "R2" in c or "RMSE" in c]))
    if SAVE_RESULTS:
        ph_results.to_csv(OUT_DIR / "ph_transform_results.csv", index=False)
        ph_tuning.astype(str).to_csv(OUT_DIR / "ph_transform_tuning.csv", index=False)
else:
    print("pH not in TARGETS, skipped.")

## 8. Feature selection inside the nested LOO (fix 2)

In the original, the reduced static set (`elevation_m + log_upa + local_slope`), `DO_BASE_FEATURES` and
`COND_BASE_FEATURES` were picked by comparing **outer**-LOO R² across feature sets (cells 111–129). New nested-LOO
runs were then reported with those sets. The held-out nodes had already influenced the choice, so those R² values
are optimistic.

Here, in every outer fold, the candidate feature set is chosen by **inner**-LOO MSE only, separately for the
linear and the graph model (graph: with scheme, λ, λ_β tuned as before, Kron Laplacians). Candidates are
chemistry + Fourier K ∈ {1, …, 5} × static ∈ {none, each of the 4 MERIT features, all 4}, so 30 per target.

The same run also gives each candidate's own nested-LOO R², which is a valid ablation table. That shows:

* **fixed baseline:** K = 1, no static (the original's "Baseline");
* **best-by-outer (biased):** the candidate with the highest outer R², i.e. what the original effectively reported;
* **selected inside (unbiased):** the honest estimate.

The daily × annual interaction terms and separate daily/annual K from the original's water-temperature section are
not among the candidates, to keep run time reasonable. In the original they changed predictions very little
(cell 92: max difference 0.009 °C).

In [ ]:
STATIC_OPTIONS = {"none": [], **{f"+ {f}": [f] for f in STATIC_FEATURES}, "+ all static": STATIC_FEATURES}
K_VALUES = [1, 2, 3, 4, 5]
SELECTION_SPACES = VARIANT_SPACES["kron_uncompressed"]
BASELINE_CANDIDATE = "K=1 | none"

data_static = node_hourly.merge(static_node_df[["node"] + STATIC_FEATURES], on="node", how="left")


def nested_loo_select(df, target, candidates, spaces, max_outer=None):
    union = sorted({f for feats in candidates.values() for f in feats})
    work = df.dropna(subset=[target] + union).copy()
    stats_union = make_node_stats(work, target, union)
    cand_stats = {name: slice_stats(stats_union, [union.index(f) for f in feats]) for name, feats in candidates.items()}
    nodes = sorted(stats_union)
    outer = nodes[:max_outer] if max_outer else nodes
    parts, tuning = [], []
    for fold, test in enumerate(outer, 1):
        t0 = time.time()
        train = [n for n in nodes if n != test]
        test_df = work[work["node"] == test]
        out = pd.DataFrame({"node": test_df["node"].to_numpy(), "y_true": test_df[target].to_numpy()})
        inner_lin, inner_gr = {}, {}
        for name, feats in candidates.items():
            stats = cand_stats[name]
            tm = train_moments(stats, train)
            alpha, lin_score = tune_linear(stats, train)
            scores = tune_graph(stats, train, spaces)
            key = best_key(scores)
            inner_lin[name], inner_gr[name] = (lin_score, alpha), (scores[key], key)
            out[f"linear|{name}"] = predict_rows(test_df, feats, fit_linear(tm, alpha))
            scheme, lam, lb = key
            out[f"graph|{name}"] = predict_rows(
                test_df, feats, solve_graph(prepare_graph(tm, lam, spaces[scheme]), lb), spaces[scheme])
        best_lin = min(inner_lin, key=lambda k: inner_lin[k][0])
        best_gr = min(inner_gr, key=lambda k: inner_gr[k][0])
        out["linear|SELECTED"] = out[f"linear|{best_lin}"]
        out["graph|SELECTED"] = out[f"graph|{best_gr}"]
        tuning.append({"target": target, "heldout_node": test, "linear_selected": best_lin,
                       "graph_selected": best_gr, "graph_scheme": inner_gr[best_gr][1][0]})
        parts.append(out)
        print(f"  {target:<21} fold {fold:>2}/{len(outer)}  linear: {best_lin:<22} graph: {best_gr:<22} "
              f"({time.time() - t0:.1f}s)", flush=True)
    preds = pd.concat(parts, ignore_index=True)
    rows = []
    for col in [c for c in preds if c not in ("node", "y_true")]:
        model, cand = col.split("|", 1)
        r2, rmse = weighted_r2_rmse(preds["y_true"].to_numpy(), preds[col].to_numpy(), preds["node"].to_numpy())
        rows.append({"Target": target, "Nodes": len(nodes), "Model": model, "Candidate": cand, "R2": r2, "RMSE": rmse})
    return pd.DataFrame(rows), pd.DataFrame(tuning)


selection_results, selection_tuning = [], []
t_start = time.time()
for target in TARGETS:
    chem = TARGET_FEATURES[target]
    candidates = {f"K={K} | {s}": chem + fourier_features(K) + extra
                  for K in K_VALUES for s, extra in STATIC_OPTIONS.items()}
    res, tun = nested_loo_select(data_static, target, candidates, SELECTION_SPACES, MAX_OUTER_FOLDS)
    selection_results.append(res)
    selection_tuning.append(tun)
selection_results = pd.concat(selection_results, ignore_index=True)
selection_tuning = pd.concat(selection_tuning, ignore_index=True)
print(f"\nDone in {time.time() - t_start:.0f}s")

In [ ]:
summary_rows = []
for (target, model), g in selection_results.groupby(["Target", "Model"], sort=False):
    per_cand = g[g["Candidate"] != "SELECTED"].set_index("Candidate")
    best_outer = per_cand["R2"].idxmax()
    summary_rows.append({
        "Target": target, "Model": model,
        "fixed baseline R2 (K=1, no static)": per_cand.loc[BASELINE_CANDIDATE, "R2"],
        "best-by-outer R2 (biased)": per_cand.loc[best_outer, "R2"],
        "best-by-outer candidate": best_outer,
        "selected-inside R2 (unbiased)": g.loc[g["Candidate"] == "SELECTED", "R2"].iloc[0],
    })
selection_summary = pd.DataFrame(summary_rows)
selection_summary["optimism (biased − unbiased)"] = (
    selection_summary["best-by-outer R2 (biased)"] - selection_summary["selected-inside R2 (unbiased)"])
display(selection_summary.style.format("{:.4f}", subset=[c for c in selection_summary if "R2" in c or "optimism" in c]))

print("How often each candidate was selected inside the inner LOO (graph model):")
display(selection_tuning.groupby("target")["graph_selected"].value_counts().rename("outer folds").to_frame())
print("Graph weighting scheme picked for the selected candidate:")
display(selection_tuning.groupby("target")["graph_scheme"].value_counts().rename("outer folds").to_frame())

if SAVE_RESULTS:
    selection_results.to_csv(OUT_DIR / "nested_selection_all_candidates.csv", index=False)
    selection_tuning.to_csv(OUT_DIR / "nested_selection_tuning.csv", index=False)
    selection_summary.to_csv(OUT_DIR / "nested_selection_summary.csv", index=False)

## 9. Summary

Built from this run's results only (fix 5: no hard-coded numbers).

In [ ]:
lin = kron_check_results[kron_check_results["Mode"] == "linear"].set_index("Target")["R2"]
st = r2_table.xs("scheme_tuned", level="Mode")
dist = r2_table.xs("distance_only", level="Mode")
sel = selection_summary.set_index(["Target", "Model"])
summary = pd.DataFrame({
    "linear, chemistry only": lin,
    "graph distance_only (Kron)": dist["kron_uncompressed"],
    "graph scheme-tuned (Kron)": st["kron_uncompressed"],
    "max |R2 change| Kron→full, any variant": r2_table[[c for c in r2_table if "−" in c]].abs()
                                                 .groupby(level="Target").max().max(axis=1),
    "linear, features selected inside": sel.xs("linear", level="Model")["selected-inside R2 (unbiased)"],
    "graph, features selected inside": sel.xs("graph", level="Model")["selected-inside R2 (unbiased)"],
}).loc[TARGETS]
display(summary.style.format("{:.4f}").format("{:.1e}", subset=["max |R2 change| Kron→full, any variant"]))
if SAVE_RESULTS:
    summary.to_csv(OUT_DIR / "summary.csv")
    print("Saved results to", OUT_DIR.relative_to(BASE_DIR))